In [1]:
#!/usr/bin/env python3
"""
LIBSTEMPO SIDE
Compute and save antenna pattern info for one pulsar in IPTA_MDC2 dataset_2.
Run this inside the Singularity container where libstempo works.
"""

import numpy as np
import libstempo as lt
import os

# ---------- CONFIG ----------
# pick any representative pulsar in dataset_2
parfile = "/scratch/na00078/projects/IPTA_MDC2/mdc2/group2/dataset_2/par/J1909-3744.par"
timfile = "/scratch/na00078/projects/IPTA_MDC2/mdc2/group2/dataset_2/tim/J1909-3744.tim"

# GW injection parameters from dataset_2
gwtheta = 0.6387905062299246
gwphi   = 3.3335788713091694
psi_inj = 1.1187560505283651
phase0_inj = 0.24434609527920614
# -----------------------------

def triad_libstempo(theta, phi):
    m  = np.array([np.sin(phi), -np.cos(phi), 0.0])
    n  = np.array([-np.cos(theta)*np.cos(phi),
                   -np.cos(theta)*np.sin(phi),
                    np.sin(theta)])
    Om = np.array([-np.sin(theta)*np.cos(phi),
                   -np.sin(theta)*np.sin(phi),
                   -np.cos(theta)])
    return m, n, Om

def phat_from_psr(psr):
    dec = psr["DECJ"].val
    ra  = psr["RAJ"].val
    ptheta = np.pi/2 - dec
    pphi   = ra
    phat = np.array([np.sin(ptheta)*np.cos(pphi),
                     np.sin(ptheta)*np.sin(pphi),
                     np.cos(ptheta)])
    return phat

def antenna_patterns(m, n, Om, phat):
    cosMu = -np.dot(Om, phat)
    denom = 1.0 - cosMu
    Fp = 0.5 * ((m@phat)**2 - (n@phat)**2) / denom
    Fx =      ((m@phat) * (n@phat)) / denom
    return Fp, Fx

# ---- compute patterns ----
psr = lt.tempopulsar(parfile, timfile)
phat = phat_from_psr(psr)
m_ls, n_ls, Om_ls = triad_libstempo(gwtheta, gwphi)
Fp_ls, Fx_ls = antenna_patterns(m_ls, n_ls, Om_ls, phat)

# ---- save for QuickCW side ----
np.save("F_libstempo.npy", np.array([Fp_ls, Fx_ls]))
np.save("phat_libstempo.npy", phat)

print("Saved F_libstempo.npy and phat_libstempo.npy")
print(f"Fp={Fp_ls:.6e}, Fx={Fx_ls:.6e}")
print(f"Injection psi={psi_inj:.6f}, phase0={phase0_inj:.6f}")


Saved F_libstempo.npy and phat_libstempo.npy
Fp=1.720202e-01, Fx=1.494091e-01
Injection psi=1.118756, phase0=0.244346
